# Setup

In [1]:
master = "spark://spark-master:7077"
dataFolder = "/data/" if master.startswith("spark://") else "../data/"

In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, LongType
from pyspark.sql import Window

spark_builder = (SparkSession.builder
         .master(master)
         .appName("spark-profiling-exercises")
         .config("spark.driver.memory", "2g")
         .config("spark.executor.memory", "1536m")   # stay under the worker's --memory 2G cap
         .config("spark.ui.showConsoleProgress", "false"))

# if master.startswith("spark://"):
#     spark_builder = (spark_builder
#                       .config("spark.driver.host", "localhost")
#                       .config("spark.driver.bindAddress", "0.0.0.0"))

spark = spark_builder.getOrCreate()

print("Spark UI:", spark.sparkContext.uiWebUrl)
print("Default parallelism (cores seen by the driver):", spark.sparkContext.defaultParallelism)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/04 21:55:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark UI: http://253ef0776d07:4040
Default parallelism (cores seen by the driver): 2


In [3]:
import sys
sys.path.append("/workspace/apps")

from spark_profiler import SparkProfiler
profiler = SparkProfiler(spark)



## Integração do M1
### Data ingestion and cleaning
Load the monthly Parquet files. Enforce the schema
(cast types, handle nulls)

In [4]:
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, IntegerType,
                               DoubleType, StringType, TimestampType)

# ---------- parâmetros ----------
months = [1, 2, 3, 4, 5, 6]
years  = [2018, 2019, 2020]

# ---------- schemas ----------
TARGET_SCHEMA = StructType([
    StructField("tpep_pickup_datetime",  TimestampType()),
    StructField("tpep_dropoff_datetime", TimestampType()),
    StructField("passenger_count",       IntegerType()),
    StructField("trip_distance",         DoubleType()),
    StructField("RatecodeID",            IntegerType()),
    StructField("PULocationID",          IntegerType()),
    StructField("DOLocationID",          IntegerType()),
    StructField("payment_type",          IntegerType()),
    StructField("fare_amount",           DoubleType()),
    StructField("tip_amount",            DoubleType()),
    StructField("total_amount",          DoubleType()),
])

ZONES_SCHEMA = StructType([
    StructField("LocationID",   IntegerType()),
    StructField("Borough",      StringType()),
    StructField("Zone",         StringType()),
    StructField("service_zone", StringType()),
])

# ---------- leitura ----------
def to_target_schema(df, schema=TARGET_SCHEMA):
    """Cast de cada coluna para o tipo alvo; colunas em falta ficam a null."""
    existing = set(df.columns)
    cols = []
    for f in schema.fields:
        src = F.col(f.name) if f.name in existing else F.lit(None)
        cols.append(src.cast(f.dataType).alias(f.name))
    return df.select(cols)

def read_year(year, months):
    """Lê todos os meses de um ano de uma vez (o schema é igual dentro do ano)."""
    paths = [dataFolder + f"yellow_tripdata_{year}-{m:02d}.parquet" for m in months]
    return (to_target_schema(spark.read.parquet(*paths))
            .withColumn("file_year", F.lit(year)))

def load_trips(years, months):
    result = read_year(years[0], months)
    for y in years[1:]:
        result = result.unionByName(read_year(y, months))
    return result

# ---------- critérios de limpeza ----------
NULL_COLS = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
             "trip_distance", "fare_amount", "PULocationID", "DOLocationID"]

def loc(c):
    return (F.col(c) < 1) | (F.col(c) > 263)

def any_null(cols):
    cond = F.col(cols[0]).isNull()
    for c in cols[1:]:
        cond = cond | F.col(c).isNull()
    return cond

INVALID = {
    "null_values":           any_null(NULL_COLS),
    "fare_amount_le_0":      F.col("fare_amount") <= 0,
    "trip_distance_le_0":    F.col("trip_distance") <= 0,
    "passenger_count_0_gt6": (F.col("passenger_count") == 0) | (F.col("passenger_count") > 6),
    "dropoff_le_pickup":     F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime"),
    "location_out_of_range": loc("PULocationID") | loc("DOLocationID"),
    "year_mismatch":         F.year("tpep_pickup_datetime") != F.col("file_year"),  # extra
}

def make_bad_flags(invalid):
    bad = {}
    for name, cond in invalid.items():
        bad[name] = F.coalesce(cond, F.lit(False))
    return bad

def any_true(conditions):
    result = conditions[0]
    for cond in conditions[1:]:
        result = result | cond
    return result

def first_failed_reason(bad):
    reason = F.lit(None).cast("string")
    for name in reversed(list(bad.keys())):
        reason = F.when(bad[name], name).otherwise(reason)
    return reason

def build_aggregations(bad, any_bad):
    aggs = [F.count("*").alias("total"),
            F.sum((~any_bad).cast("int")).alias("kept")]
    for name, cond in bad.items():
        aggs.append(F.sum(cond.cast("int")).alias(f"fail_{name}"))
    for name in bad.keys():
        aggs.append(F.sum((F.col("reason") == name).cast("int")).alias(f"first_{name}"))
    return aggs

def cleaning_stats(trips, bad, any_bad):
    return (trips
        .withColumn("reason", first_failed_reason(bad))
        .groupBy("file_year")
        .agg(*build_aggregations(bad, any_bad)))      # sem orderBy: ordena-se no pandas

def clean_trips(trips, any_bad):
    return trips.filter(~any_bad).drop("file_year")

Leitura


In [5]:
with profiler.profile("M1-read"):
    trips = load_trips(years, months)
    zones = spark.read.csv(dataFolder + "taxi_zone_lookup.csv",
                           header=True, schema=ZONES_SCHEMA)

[M1-read] wall time: 1.92s (3 stages, 3 tasks) - see profiler.report() / profiler.compare(...)


limpeza


In [6]:
with profiler.profile("M1-clean&stats"):
    bad     = make_bad_flags(INVALID)
    any_bad = any_true(list(bad.values()))
    stats       = cleaning_stats(trips, bad, any_bad)
    trips_typed = clean_trips(trips, any_bad)
    stats_pdf = stats.toPandas().sort_values("file_year")

display(stats_pdf.set_index("file_year").T)
display(profiler.report())

/opt/spark/python/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


[M1-clean&stats] wall time: 10.91s (3 stages, 37 tasks) - see profiler.report() / profiler.compare(...)


file_year,2018,2019,2020
total,53930466,44658561,16848347
kept,52261062,42743052,15858367
fail_null_values,0,197782,281282
fail_fare_amount_le_0,44230,85489,63308
fail_trip_distance_le_0,345392,329087,195970
fail_passenger_count_0_gt6,402625,785857,330958
fail_dropoff_le_pickup,45052,42124,12041
fail_location_out_of_range,1025292,676572,185819
fail_year_mismatch,748,742,220
first_null_values,0,197782,281282


,wall_time_s,num_jobs,num_stages,num_tasks,executor_run_time_ms,executor_cpu_time_ms,input_bytes,output_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes
run,,,,,,,,,,,,
M1-read,1.92,3,3,3,370,106,0.0B,0.0B,0.0B,0.0B,0.0B,0.0B
M1-clean&stats,10.91,2,3,37,50217,38834,252.2KB,0.0B,2.2KB,2.2KB,0.0B,0.0B


In [7]:
from functools import reduce

from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, TimestampType,
                               IntegerType, DoubleType)

# =============================================================
# 1. CONFIGURAÇÃO  (fora do profiler: não é trabalho do Spark)
# =============================================================
MONTHS = [1, 2, 3, 4, 5, 6]
YEARS  = [2018, 2019, 2020]
FILES  = [(dataFolder + f"yellow_tripdata_{y}-{m:02d}.parquet", y)
          for y in YEARS for m in MONTHS]

TARGET_SCHEMA = StructType([
    StructField("tpep_pickup_datetime",  TimestampType()),
    StructField("tpep_dropoff_datetime", TimestampType()),
    StructField("passenger_count",       IntegerType()),
    StructField("trip_distance",         DoubleType()),
    StructField("RatecodeID",            IntegerType()),
    StructField("PULocationID",          IntegerType()),
    StructField("DOLocationID",          IntegerType()),
    StructField("payment_type",          IntegerType()),
    StructField("fare_amount",           DoubleType()),
    StructField("tip_amount",            DoubleType()),
    StructField("total_amount",          DoubleType()),
])

NULL_COLS = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
             "trip_distance", "fare_amount", "PULocationID", "DOLocationID"]


# =============================================================
# 2. LEITURA
# =============================================================
def to_target_schema(df, schema=TARGET_SCHEMA):
    """Alinha cada ficheiro ao schema alvo (colunas em falta ficam null)."""
    existing = set(df.columns)
    return df.select([
        (F.col(f.name) if f.name in existing else F.lit(None))
            .cast(f.dataType).alias(f.name)
        for f in schema.fields
    ])


def read_month(path, year):
    return (to_target_schema(spark.read.parquet(path))
            .withColumn("file_year", F.lit(year)))


def load_trips(files):
    return reduce(lambda a, b: a.unionByName(b),
                  [read_month(p, y) for p, y in files])


# =============================================================
# 3. CRITÉRIOS DE LIMPEZA
# =============================================================
def any_true(conds):
    """Junta condições com OU."""
    return reduce(lambda a, b: a | b, conds)


def loc_out_of_range(c):
    return (F.col(c) < 1) | (F.col(c) > 263)


INVALID = {
    "null_values":           any_true([F.col(c).isNull() for c in NULL_COLS]),
    "fare_amount_le_0":      F.col("fare_amount") <= 0,
    "trip_distance_le_0":    F.col("trip_distance") <= 0,
    "passenger_count_0_gt6": (F.col("passenger_count") == 0) | (F.col("passenger_count") > 6),
    "dropoff_le_pickup":     F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime"),
    "location_out_of_range": loc_out_of_range("PULocationID") | loc_out_of_range("DOLocationID"),
    "year_mismatch":         F.year("tpep_pickup_datetime") != F.col("file_year"),
}

# null -> False: comparações com null não contam como falha
# (os nulls já são apanhados por 'null_values')
BAD     = {name: F.coalesce(cond, F.lit(False)) for name, cond in INVALID.items()}
ANY_BAD = any_true(list(BAD.values()))


def first_failed_reason(bad):
    """CASE WHEN c1 THEN 'c1' WHEN c2 THEN 'c2' ... ELSE null END"""
    reason = F.lit(None).cast("string")
    for name in reversed(list(bad)):
        reason = F.when(bad[name], name).otherwise(reason)
    return reason


# =============================================================
# 4. ESTATÍSTICAS E LIMPEZA
# =============================================================
def cleaning_stats(trips, bad=BAD, any_bad=ANY_BAD):
    """Estatísticas por ano num só scan:
    total, válidos, falhas por critério (sobrepostas)
    e falhas pelo primeiro critério (sequenciais)."""
    aggs = [F.count("*").alias("total"),
            F.sum((~any_bad).cast("int")).alias("kept")]
    aggs += [F.sum(c.cast("int")).alias(f"fail_{n}") for n, c in bad.items()]
    aggs += [F.sum((F.col("reason") == n).cast("int")).alias(f"first_{n}") for n in bad]

    return (trips
            .withColumn("reason", first_failed_reason(bad))
            .groupBy("file_year")
            .agg(*aggs)
            .orderBy("file_year"))


def clean_trips(trips, any_bad=ANY_BAD):
    """Mantém só os registos válidos."""
    return trips.filter(~any_bad).drop("file_year")


# =============================================================
# 5. EXECUÇÃO  (só isto conta para o profiler)
# =============================================================
with profiler.profile("sugestao de melhora"):
    trips = load_trips(FILES)

    # Action 1: estatísticas de limpeza
    stats_pdf = cleaning_stats(trips).toPandas()

    # Action 2: escrita dos dados limpos, particionados por ano/mês
    (clean_trips(trips)
        .withColumn("year",  F.year("tpep_pickup_datetime"))
        .withColumn("month", F.month("tpep_pickup_datetime"))
        .repartition("year", "month")          # 1 ficheiro por pasta year/month
        .write.mode("overwrite")
        .option("parquet.block.size", 32 * 1024 * 1024)
        .partitionBy("year", "month")
        .parquet(dataFolder + "trips_clean"))

profiler.report()
stats_pdf.T

/opt/spark/python/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


[sugestao de melhora] wall time: 74.03s (29 stages, 622 tasks) - see profiler.report() / profiler.compare(...)


,0,1,2
file_year,2018,2019,2020
total,53930466,44658561,16848347
kept,52261062,42743052,15858367
fail_null_values,0,197782,281282
fail_fare_amount_le_0,44230,85489,63308
fail_trip_distance_le_0,345392,329087,195970
fail_passenger_count_0_gt6,402625,785857,330958
fail_dropoff_le_pickup,45052,42124,12041
fail_location_out_of_range,1025292,676572,185819
fail_year_mismatch,748,742,220


# M2

In [8]:
with profiler.profile("2-temporal-demand-analysis"):
    trips2 = (trips_typed
        .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
        .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime"))
        .withColumn("duration_min",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60))

    joined = (trips2
        .join(zones, trips2.PULocationID == zones.LocationID, "left")
        .withColumnRenamed("Borough", "pickup_borough"))

    m2 = (joined
        .groupBy("pickup_borough", "day_of_week", "hour_of_day")
        .agg(
            F.count("*").alias("trip_count"),
            F.avg("fare_amount").alias("avg_fare"),
            F.avg("duration_min").alias("avg_duration_min"),
            F.avg("trip_distance").alias("avg_distance"),
        ))

    m2.count()

    m2.orderBy("pickup_borough", "day_of_week", "hour_of_day").show(20)

    (m2.write
        .mode("overwrite")
        .partitionBy("pickup_borough")
        .parquet("/data/output/m2_temporal_demand"))



+--------------+-----------+-----------+----------+------------------+------------------+------------------+
|pickup_borough|day_of_week|hour_of_day|trip_count|          avg_fare|  avg_duration_min|      avg_distance|
+--------------+-----------+-----------+----------+------------------+------------------+------------------+
|         Bronx|          1|          0|       852|16.939330985915493| 16.99643974960876| 4.452746478873239|
|         Bronx|          1|          1|       718|15.632325905292479|25.138324048282264| 4.128398328690808|
|         Bronx|          1|          2|       631|14.042282091917592|15.611621764395142|3.8217749603803486|
|         Bronx|          1|          3|       677|13.626011816838995| 17.61085672082718|3.6337370753323484|
|         Bronx|          1|          4|       728|14.591868131868134| 19.17797619047619|4.0161401098901095|
|         Bronx|          1|          5|       544|15.449283088235294|11.838541666666668| 4.517591911764705|
|         Bronx|   

# Integração M3
## Zone-level statistics
For each pickup zone and month, compute mean and
standard deviation of fare_amount and trip_distance


In [15]:
trips_typed.printSchema()
zones.printSchema()

root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [9]:
with (profiler.profile("M3- Zone Level statistics")):

    joined = (trips_typed
            .join(zones, trips_typed.PULocationID == zones.LocationID))

    monthly = (joined
    .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime"))
    .groupBy("Borough", "Zone", "month")
    .agg(F.count("*").alias("trip_count")))
    w = Window.partitionBy("Borough", "month").orderBy(F.desc("trip_count"))

    ranked = monthly.withColumn("rank", F.dense_rank().over(w))

    ranked.orderBy("Borough", "month", "rank").show()


profiler.report()



+-------+--------------------+-------------------+----------+----+
|Borough|                Zone|              month|trip_count|rank|
+-------+--------------------+-------------------+----------+----+
|  Bronx|Mott Haven/Port M...|2018-01-01 00:00:00|      1296|   1|
|  Bronx|      West Concourse|2018-01-01 00:00:00|      1113|   2|
|  Bronx|       Melrose South|2018-01-01 00:00:00|       455|   3|
|  Bronx|          Highbridge|2018-01-01 00:00:00|       365|   4|
|  Bronx|East Concourse/Co...|2018-01-01 00:00:00|       354|   5|
|  Bronx|University Height...|2018-01-01 00:00:00|       250|   6|
|  Bronx|  Morrisania/Melrose|2018-01-01 00:00:00|       174|   7|
|  Bronx|Spuyten Duyvil/Ki...|2018-01-01 00:00:00|       162|   8|
|  Bronx|         Parkchester|2018-01-01 00:00:00|       140|   9|
|  Bronx|         Hunts Point|2018-01-01 00:00:00|       140|   9|
|  Bronx|          Mount Hope|2018-01-01 00:00:00|       133|  10|
|  Bronx|  Soundview/Bruckner|2018-01-01 00:00:00|       125| 

,wall_time_s,num_jobs,num_stages,num_tasks,executor_run_time_ms,executor_cpu_time_ms,input_bytes,output_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes
run,,,,,,,,,,,,
M1-read,1.92,3,3,3,370,106,0.0B,0.0B,0.0B,0.0B,0.0B,0.0B
M1-clean&stats,10.91,2,3,37,50217,38834,252.2KB,0.0B,2.2KB,2.2KB,0.0B,0.0B
sugestao de melhora,74.03,24,29,622,375215,314345,1.9MB,2.0GB,4.2GB,4.2GB,9.9GB,3.5GB
2-temporal-demand-analysis,46.29,10,15,134,235837,218701,792.6KB,39.3KB,2.3MB,2.3MB,0.0B,0.0B
M3- Zone Level statistics,15.20,4,7,58,76453,72411,264.2KB,0.0B,596.6KB,596.6KB,0.0B,0.0B


In [10]:
with (profiler.profile("M3- Zone Level statistics pruning")):

    trips_m3 = trips_typed.select("PULocationID", "tpep_pickup_datetime", "DOLocationID")
    zones_m3 = zones.select("LocationID", "Borough", "Zone")

    joined = trips_m3.join(zones_m3, trips_m3.PULocationID == zones_m3.LocationID)

    monthly = (joined
    .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime"))
    .groupBy("Borough", "Zone", "month")
    .agg(F.count("*").alias("trip_count")))
    w = Window.partitionBy("Borough", "month").orderBy(F.desc("trip_count"))
    ranked = monthly.withColumn("rank", F.dense_rank().over(w))
    ranked.orderBy("Borough", "month", "rank").show()

    ##### quota
    m = monthly.withColumn("month_idx", F.year("month") * 12 + F.month("month"))


    w_zone = (Window.partitionBy("Borough", "Zone")
                    .orderBy("month_idx")
                    .rangeBetween(-2, 0))

    zone_roll = m.withColumn("zone_3m", F.sum("trip_count").over(w_zone))

    borough_month = (m.groupBy("Borough", "month_idx")
                      .agg(F.sum("trip_count").alias("borough_count")))


    w_borough = (Window.partitionBy("Borough")
                       .orderBy("month_idx")
                       .rangeBetween(-2, 0))

    borough_roll = (borough_month
        .withColumn("borough_3m", F.sum("borough_count").over(w_borough))
        .select("Borough", "month_idx", "borough_3m"))


    share = (zone_roll
        .join(borough_roll, ["Borough", "month_idx"])
        .withColumn("share_3m", F.round(F.col("zone_3m") / F.col("borough_3m"), 4))
        .select("Borough", "Zone", "month", "trip_count", "zone_3m", "borough_3m", "share_3m"))

    share.orderBy("Borough", "Zone", "month").show(20, truncate=False)



profiler.report()



+-------+--------------------+-------------------+----------+----+
|Borough|                Zone|              month|trip_count|rank|
+-------+--------------------+-------------------+----------+----+
|  Bronx|Mott Haven/Port M...|2018-01-01 00:00:00|      1296|   1|
|  Bronx|      West Concourse|2018-01-01 00:00:00|      1113|   2|
|  Bronx|       Melrose South|2018-01-01 00:00:00|       455|   3|
|  Bronx|          Highbridge|2018-01-01 00:00:00|       365|   4|
|  Bronx|East Concourse/Co...|2018-01-01 00:00:00|       354|   5|
|  Bronx|University Height...|2018-01-01 00:00:00|       250|   6|
|  Bronx|  Morrisania/Melrose|2018-01-01 00:00:00|       174|   7|
|  Bronx|Spuyten Duyvil/Ki...|2018-01-01 00:00:00|       162|   8|
|  Bronx|         Parkchester|2018-01-01 00:00:00|       140|   9|
|  Bronx|         Hunts Point|2018-01-01 00:00:00|       140|   9|
|  Bronx|          Mount Hope|2018-01-01 00:00:00|       133|  10|
|  Bronx|  Soundview/Bruckner|2018-01-01 00:00:00|       125| 

,wall_time_s,num_jobs,num_stages,num_tasks,executor_run_time_ms,executor_cpu_time_ms,input_bytes,output_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes
run,,,,,,,,,,,,
M1-read,1.92,3,3,3,370,106,0.0B,0.0B,0.0B,0.0B,0.0B,0.0B
M1-clean&stats,10.91,2,3,37,50217,38834,252.2KB,0.0B,2.2KB,2.2KB,0.0B,0.0B
sugestao de melhora,74.03,24,29,622,375215,314345,1.9MB,2.0GB,4.2GB,4.2GB,9.9GB,3.5GB
2-temporal-demand-analysis,46.29,10,15,134,235837,218701,792.6KB,39.3KB,2.3MB,2.3MB,0.0B,0.0B
M3- Zone Level statistics,15.20,4,7,58,76453,72411,264.2KB,0.0B,596.6KB,596.6KB,0.0B,0.0B
M3- Zone Level statistics pruning,35.63,11,20,122,182859,161157,528.4KB,0.0B,1.6MB,1.1MB,0.0B,0.0B


In [11]:
with (profiler.profile("M3- Zone Level statistics pruning + quota optimization")):
    trips_m3 = trips_typed.select("PULocationID", "tpep_pickup_datetime", "DOLocationID")
    zones_m3 = zones.select("LocationID", "Borough", "Zone")

    joined = trips_m3.join(zones_m3, trips_m3.PULocationID == zones_m3.LocationID)

    monthly = (joined
    .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime"))
    .groupBy("Borough", "Zone", "month")
    .agg(F.count("*").alias("trip_count"))
    .cache())
    w = Window.partitionBy("Borough", "month").orderBy(F.desc("trip_count"))
    ranked = monthly.withColumn("rank", F.dense_rank().over(w))
    ranked.orderBy("Borough", "month", "rank").show()

    ##### quota
    m = monthly.withColumn("month_idx", F.year("month") * 12 + F.month("month"))


    w_zone = (Window.partitionBy("Borough", "Zone")
                    .orderBy("month_idx")
                    .rangeBetween(-2, 0))

    zone_roll = m.withColumn("zone_3m", F.sum("trip_count").over(w_zone))

    borough_month = (m.groupBy("Borough", "month_idx")
                      .agg(F.sum("trip_count").alias("borough_count")))


    w_borough = (Window.partitionBy("Borough")
                       .orderBy("month_idx")
                       .rangeBetween(-2, 0))

    borough_roll = (borough_month
        .withColumn("borough_3m", F.sum("borough_count").over(w_borough))
        .select("Borough", "month_idx", "borough_3m"))


    share = (zone_roll
        .join(borough_roll, ["Borough", "month_idx"])
        .withColumn("share_3m", F.round(F.col("zone_3m") / F.col("borough_3m"), 4))
        .select("Borough", "Zone", "month", "trip_count", "zone_3m", "borough_3m", "share_3m"))

    share.orderBy("Borough", "Zone", "month").show(20, truncate=False)

    w_prev = (Window.partitionBy("Borough", "Zone")
                    .orderBy("month_idx")
                    .rangeBetween(-3, -1))

    anomalies = (m
        .withColumn("roll_mean", F.avg("trip_count").over(w_prev))
        .withColumn("roll_std",  F.stddev_samp("trip_count").over(w_prev))
        .withColumn("roll_n",    F.count("trip_count").over(w_prev))
        .withColumn("z_score",
            F.when((F.col("roll_n") >= 2) & (F.col("roll_std") > 0),
                   (F.col("trip_count") - F.col("roll_mean")) / F.col("roll_std")))
        .withColumn("is_anomaly", F.coalesce(F.abs("z_score") > 1.5, F.lit(False)))
        .select("Borough", "Zone", "month", "trip_count",
                F.round("roll_mean", 1).alias("roll_mean"),
                F.round("roll_std", 1).alias("roll_std"),
                F.round("z_score", 2).alias("z_score"),
                "is_anomaly"))

    (anomalies
        .filter("is_anomaly")
        .orderBy(F.desc(F.abs("z_score")))
        .show(20, truncate=False))

+-------+--------------------+-------------------+----------+----+
|Borough|                Zone|              month|trip_count|rank|
+-------+--------------------+-------------------+----------+----+
|  Bronx|Mott Haven/Port M...|2018-01-01 00:00:00|      1296|   1|
|  Bronx|      West Concourse|2018-01-01 00:00:00|      1113|   2|
|  Bronx|       Melrose South|2018-01-01 00:00:00|       455|   3|
|  Bronx|          Highbridge|2018-01-01 00:00:00|       365|   4|
|  Bronx|East Concourse/Co...|2018-01-01 00:00:00|       354|   5|
|  Bronx|University Height...|2018-01-01 00:00:00|       250|   6|
|  Bronx|  Morrisania/Melrose|2018-01-01 00:00:00|       174|   7|
|  Bronx|Spuyten Duyvil/Ki...|2018-01-01 00:00:00|       162|   8|
|  Bronx|         Parkchester|2018-01-01 00:00:00|       140|   9|
|  Bronx|         Hunts Point|2018-01-01 00:00:00|       140|   9|
|  Bronx|          Mount Hope|2018-01-01 00:00:00|       133|  10|
|  Bronx|  Soundview/Bruckner|2018-01-01 00:00:00|       125| 

# O1 — Fare anomaly detection.
## For each (PULocationID, hour_of_day) bucket,

compute mean and σ of fare_amount / trip_distance .
flag trips where the per-km fare exceeds mean + 3σ.
compare weekday vs. weekend anomaly rates broken down by payment_type and pickup
borough.

(Mais rapido)

In [9]:
from pyspark.sql import Window

w = Window.partitionBy("PULocationID", "hour_of_day")

with profiler.profile("O1-fare-anomaly-window"):
    flagged = (trips_typed
        .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
        .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
        .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime"))
        .withColumn("fpm_mean", F.avg("fare_per_mile").over(w))
        .withColumn("fpm_std",  F.stddev("fare_per_mile").over(w))
        .withColumn("is_anomaly", F.col("fare_per_mile") > F.col("fpm_mean") + 3 * F.col("fpm_std"))
        .withColumn("is_weekend", F.col("day_of_week").isin(1, 7)))

    flagged = (flagged
        .join(zones, flagged.PULocationID == zones.LocationID, "left")
        .withColumnRenamed("Borough", "pickup_borough"))

    o1 = (flagged
        .groupBy("pickup_borough", "payment_type", "is_weekend")
        .agg(F.count("*").alias("trips"),
             F.avg(F.col("is_anomaly").cast("int")).alias("anomaly_rate")))

    o1.count()

o1.orderBy("pickup_borough", "payment_type", "is_weekend").show(50)


[O1-fare-anomaly-window] wall time: 4.41s (7 stages, 22 tasks) - see profiler.report() / profiler.compare(...)
+--------------+------------+----------+-------+--------------------+
|pickup_borough|payment_type|is_weekend|  trips|        anomaly_rate|
+--------------+------------+----------+-------+--------------------+
|         Bronx|           1|     false|   5166|0.009697439875872769|
|         Bronx|           1|      true|   1747|0.009221902017291067|
|         Bronx|           2|     false|   5749|0.026067179846046187|
|         Bronx|           2|      true|   2333|0.023686477174849268|
|         Bronx|           3|     false|    102|  0.0784313725490196|
|         Bronx|           3|      true|     55| 0.07272727272727272|
|         Bronx|           4|     false|     45| 0.06666666666666667|
|         Bronx|           4|      true|     24|0.041666666666666664|
|      Brooklyn|           1|     false|  59410|0.006365243748421...|
|      Brooklyn|           1|      true|  25734| 

(Mais lento)

In [10]:
with profiler.profile("O1-fare-anomaly"):
    o1_base = (trips_typed
        .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
        .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
        .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime")))

    stats = (o1_base
        .groupBy("PULocationID", "hour_of_day")
        .agg(F.avg("fare_per_mile").alias("fpm_mean"),
             F.stddev("fare_per_mile").alias("fpm_std")))

    flagged = (o1_base
        .join(stats, ["PULocationID", "hour_of_day"])
        .withColumn("is_anomaly",
                    F.col("fare_per_mile") > F.col("fpm_mean") + 3 * F.col("fpm_std")))

    flagged = flagged.withColumn("is_weekend", F.col("day_of_week").isin(1, 7))

    flagged = (flagged
        .join(zones, flagged.PULocationID == zones.LocationID, "left")
        .withColumnRenamed("Borough", "pickup_borough"))

    o1 = (flagged
        .groupBy("pickup_borough", "payment_type", "is_weekend")
        .agg(F.count("*").alias("trips"),
             F.avg(F.col("is_anomaly").cast("int")).alias("anomaly_rate")))

    o1.count()

o1.orderBy("pickup_borough", "payment_type", "is_weekend").show(50)
profiler.compare("O1-fare-anomaly-window", "O1-fare-anomaly")

[O1-fare-anomaly] wall time: 8.35s (14 stages, 59 tasks) - see profiler.report() / profiler.compare(...)
+--------------+------------+----------+-------+--------------------+
|pickup_borough|payment_type|is_weekend|  trips|        anomaly_rate|
+--------------+------------+----------+-------+--------------------+
|         Bronx|           1|     false|   5166|0.009697439875872769|
|         Bronx|           1|      true|   1747|0.009221902017291067|
|         Bronx|           2|     false|   5749|0.026067179846046187|
|         Bronx|           2|      true|   2333|0.023686477174849268|
|         Bronx|           3|     false|    102|  0.0784313725490196|
|         Bronx|           3|      true|     55| 0.07272727272727272|
|         Bronx|           4|     false|     45| 0.06666666666666667|
|         Bronx|           4|      true|     24|0.041666666666666664|
|      Brooklyn|           1|     false|  59410|0.006365243748421...|
|      Brooklyn|           1|      true|  25734| 0.0059

,O1-fare-anomaly-window,O1-fare-anomaly,change
metric,,,
wall_time_s,4.41,8.35,+89.3%
num_jobs,4,7,+75.0%
num_stages,7,14,+100.0%
num_tasks,22,59,+168.2%
executor_run_time_ms,10423,30714,+194.7%
executor_cpu_time_ms,9857,29377,+198.0%
input_bytes,753.0KB,1.5MB,+98.4%
output_bytes,0.0B,0.0B,-
shuffle_read_bytes,11.3KB,33.5MB,+303353.8%


Versão com persist

In [11]:
from pyspark import StorageLevel

with profiler.profile("O1-fare-anomaly-persist"):
    o1_base = (trips_typed
        .select("PULocationID", "payment_type", "fare_amount",
                "trip_distance", "tpep_pickup_datetime")
        .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
        .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
        .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime"))
        .persist(StorageLevel.MEMORY_AND_DISK))

    stats = (o1_base
        .groupBy("PULocationID", "hour_of_day")
        .agg(F.avg("fare_per_mile").alias("fpm_mean"),
             F.stddev("fare_per_mile").alias("fpm_std")))

    flagged = (o1_base
        .join(stats, ["PULocationID", "hour_of_day"])
        .withColumn("is_anomaly",
                    F.col("fare_per_mile") > F.col("fpm_mean") + 3 * F.col("fpm_std"))
        .withColumn("is_weekend", F.col("day_of_week").isin(1, 7)))

    flagged = (flagged
        .join(zones, flagged.PULocationID == zones.LocationID, "left")
        .withColumnRenamed("Borough", "pickup_borough"))

    o1 = (flagged
        .groupBy("pickup_borough", "payment_type", "is_weekend")
        .agg(F.count("*").alias("trips"),
             F.avg(F.col("is_anomaly").cast("int")).alias("anomaly_rate")))

    o1.count()

o1_base.unpersist()
o1.orderBy("pickup_borough", "payment_type", "is_weekend").show(50)
profiler.compare("O1-fare-anomaly-persist", "O1-fare-anomaly")

[O1-fare-anomaly-persist] wall time: 12.33s (16 stages, 71 tasks) - see profiler.report() / profiler.compare(...)
+--------------+------------+----------+-------+--------------------+
|pickup_borough|payment_type|is_weekend|  trips|        anomaly_rate|
+--------------+------------+----------+-------+--------------------+
|         Bronx|           1|     false|   5166|0.009697439875872769|
|         Bronx|           1|      true|   1747|0.009221902017291067|
|         Bronx|           2|     false|   5749|0.026067179846046187|
|         Bronx|           2|      true|   2333|0.023686477174849268|
|         Bronx|           3|     false|    102|  0.0784313725490196|
|         Bronx|           3|      true|     55| 0.07272727272727272|
|         Bronx|           4|     false|     45| 0.06666666666666667|
|         Bronx|           4|      true|     24|0.041666666666666664|
|      Brooklyn|           1|     false|  59410|0.006365243748421...|
|      Brooklyn|           1|      true|  2573

,O1-fare-anomaly-persist,O1-fare-anomaly,change
metric,,,
wall_time_s,12.33,8.35,-32.3%
num_jobs,9,7,-22.2%
num_stages,16,14,-12.5%
num_tasks,71,59,-16.9%
executor_run_time_ms,52875,30714,-41.9%
executor_cpu_time_ms,44231,29377,-33.6%
input_bytes,642.4MB,1.5MB,-99.8%
output_bytes,0.0B,0.0B,-
shuffle_read_bytes,33.5MB,33.5MB,+0.0%


Não valeu a pena, quase acabou com o meu pc e nem foi mais rapido

### Versão sem o Broadcast join, para comparar no relatorio

In [12]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold", -1)

with profiler.profile("O1-no-broadcast"):
    o1_base = (trips_typed
        .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
        .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
        .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime")))

    stats = (o1_base
        .groupBy("PULocationID", "hour_of_day")
        .agg(F.avg("fare_per_mile").alias("fpm_mean"),
             F.stddev("fare_per_mile").alias("fpm_std")))

    flagged = (o1_base
        .join(stats, ["PULocationID", "hour_of_day"])
        .withColumn("is_anomaly",
                    F.col("fare_per_mile") > F.col("fpm_mean") + 3 * F.col("fpm_std"))
        .withColumn("is_weekend", F.col("day_of_week").isin(1, 7)))

    flagged = (flagged
        .join(zones, flagged.PULocationID == zones.LocationID, "left")
        .withColumnRenamed("Borough", "pickup_borough"))

    o1 = (flagged
        .groupBy("pickup_borough", "payment_type", "is_weekend")
        .agg(F.count("*").alias("trips"),
             F.avg(F.col("is_anomaly").cast("int")).alias("anomaly_rate")))

    o1.count()

o1.explain()

spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
spark.conf.unset("spark.sql.adaptive.autoBroadcastJoinThreshold")

o1.orderBy("pickup_borough", "payment_type", "is_weekend").show(50)

profiler.compare("O1-fare-anomaly", "O1-no-broadcast")

[O1-no-broadcast] wall time: 9.74s (24 stages, 112 tasks) - see profiler.report() / profiler.compare(...)
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[pickup_borough#1433, payment_type#9L, is_weekend#1432], functions=[count(1), avg(cast(is_anomaly#1431 as int))])
   +- Exchange hashpartitioning(pickup_borough#1433, payment_type#9L, is_weekend#1432, 200), ENSURE_REQUIREMENTS, [plan_id=3915]
      +- HashAggregate(keys=[pickup_borough#1433, payment_type#9L, is_weekend#1432], functions=[partial_count(1), partial_avg(cast(is_anomaly#1431 as int))])
         +- Project [payment_type#9L, is_anomaly#1431, is_weekend#1432, Borough#37 AS pickup_borough#1433]
            +- SortMergeJoin [PULocationID#46], [LocationID#36], LeftOuter
               :- Sort [PULocationID#46 ASC NULLS FIRST], false, 0
               :  +- Exchange hashpartitioning(PULocationID#46, 200), ENSURE_REQUIREMENTS, [plan_id=3907]
               :     +- Project [PULocationID#46, payment_ty

,O1-fare-anomaly,O1-no-broadcast,change
metric,,,
wall_time_s,8.35,9.74,+16.6%
num_jobs,7,7,+0.0%
num_stages,14,24,+71.4%
num_tasks,59,112,+89.8%
executor_run_time_ms,30714,39200,+27.6%
executor_cpu_time_ms,29377,38030,+29.5%
input_bytes,1.5MB,1.5MB,+0.0%
output_bytes,0.0B,0.0B,-
shuffle_read_bytes,33.5MB,59.9MB,+78.7%


### Verificar com varias partições

In [13]:
def run_o1(df):
    o1_base = (df
        .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
        .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
        .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime")))
    stats = (o1_base.groupBy("PULocationID", "hour_of_day")
        .agg(F.avg("fare_per_mile").alias("fpm_mean"),
             F.stddev("fare_per_mile").alias("fpm_std")))
    flagged = (o1_base.join(stats, ["PULocationID", "hour_of_day"])
        .withColumn("is_anomaly", F.col("fare_per_mile") > F.col("fpm_mean") + 3 * F.col("fpm_std"))
        .withColumn("is_weekend", F.col("day_of_week").isin(1, 7)))
    flagged = (flagged.join(zones, flagged.PULocationID == zones.LocationID, "left")
        .withColumnRenamed("Borough", "pickup_borough"))
    return (flagged.groupBy("pickup_borough", "payment_type", "is_weekend")
        .agg(F.count("*").alias("trips"),
             F.avg(F.col("is_anomaly").cast("int")).alias("anomaly_rate")))

spark.conf.set("spark.sql.adaptive.enabled", "false")

for n in [6, 12, 50, 200, 1000]:
    spark.conf.set("spark.sql.shuffle.partitions", n)
    with profiler.profile(f"O1-partitions-{n}"):
        run_o1(trips_typed).count()

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", 200)

profiler.report()

[O1-partitions-6] wall time: 9.28s (6 stages, 26 tasks) - see profiler.report() / profiler.compare(...)
[O1-partitions-12] wall time: 7.95s (6 stages, 38 tasks) - see profiler.report() / profiler.compare(...)
[O1-partitions-50] wall time: 7.73s (6 stages, 114 tasks) - see profiler.report() / profiler.compare(...)
[O1-partitions-200] wall time: 8.69s (6 stages, 414 tasks) - see profiler.report() / profiler.compare(...)
[O1-partitions-1000] wall time: 13.11s (6 stages, 2014 tasks) - see profiler.report() / profiler.compare(...)


,wall_time_s,num_jobs,num_stages,num_tasks,executor_run_time_ms,executor_cpu_time_ms,input_bytes,output_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes
run,,,,,,,,,,,,
1-ingestion-cleaning,5.79,4,6,26,11937,9273,841.9KB,0.0B,696.0B,696.0B,0.0B,0.0B
2-temporal-demand-analysis,15.86,10,15,50,35844,33629,2.2MB,36.1KB,563.2KB,563.2KB,0.0B,0.0B
M3- Zone Level statistics,0.07,0,0,0,0,0,0.0B,0.0B,0.0B,0.0B,0.0B,0.0B
O1-fare-anomaly-window,4.41,4,7,22,10423,9857,753.0KB,0.0B,11.3KB,11.3KB,0.0B,0.0B
O1-fare-anomaly,8.35,7,14,59,30714,29377,1.5MB,0.0B,33.5MB,33.5MB,0.0B,0.0B
O1-fare-anomaly-persist,12.33,9,16,71,52875,44231,642.4MB,0.0B,33.5MB,33.5MB,0.0B,0.0B
O1-no-broadcast,9.74,7,24,112,39200,38030,1.5MB,0.0B,59.9MB,59.9MB,128.0MB,5.0MB
O1-partitions-6,9.28,2,6,26,33263,32488,1.5MB,0.0B,69.9MB,69.9MB,0.0B,0.0B
O1-partitions-12,7.95,2,6,38,29928,29222,1.5MB,0.0B,62.9MB,62.9MB,0.0B,0.0B


### Select

In [14]:
with profiler.profile("O1-select-cols"):
    o1_base = (trips_typed
        .select("PULocationID", "payment_type", "fare_amount",
                "trip_distance", "tpep_pickup_datetime")
        .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
        .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
        .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime")))

    stats = (o1_base
        .groupBy("PULocationID", "hour_of_day")
        .agg(F.avg("fare_per_mile").alias("fpm_mean"),
             F.stddev("fare_per_mile").alias("fpm_std")))

    flagged = (o1_base
        .join(stats, ["PULocationID", "hour_of_day"])
        .withColumn("is_anomaly",
                    F.col("fare_per_mile") > F.col("fpm_mean") + 3 * F.col("fpm_std"))
        .withColumn("is_weekend", F.col("day_of_week").isin(1, 7)))

    flagged = (flagged
        .join(zones.select("LocationID", "Borough"),
              flagged.PULocationID == zones.LocationID, "left")
        .withColumnRenamed("Borough", "pickup_borough"))

    o1 = (flagged
        .groupBy("pickup_borough", "payment_type", "is_weekend")
        .agg(F.count("*").alias("trips"),
             F.avg(F.col("is_anomaly").cast("int")).alias("anomaly_rate")))

    o1.count()

o1.orderBy("pickup_borough", "payment_type", "is_weekend").show(50)

[O1-select-cols] wall time: 6.90s (14 stages, 59 tasks) - see profiler.report() / profiler.compare(...)
+--------------+------------+----------+-------+--------------------+
|pickup_borough|payment_type|is_weekend|  trips|        anomaly_rate|
+--------------+------------+----------+-------+--------------------+
|         Bronx|           1|     false|   5166|0.009697439875872769|
|         Bronx|           1|      true|   1747|0.009221902017291067|
|         Bronx|           2|     false|   5749|0.026067179846046187|
|         Bronx|           2|      true|   2333|0.023686477174849268|
|         Bronx|           3|     false|    102|  0.0784313725490196|
|         Bronx|           3|      true|     55| 0.07272727272727272|
|         Bronx|           4|     false|     45| 0.06666666666666667|
|         Bronx|           4|      true|     24|0.041666666666666664|
|      Brooklyn|           1|     false|  59410|0.006365243748421...|
|      Brooklyn|           1|      true|  25734| 0.00598

### Window com o select, mistura

In [15]:
from pyspark.sql import Window

w = Window.partitionBy("PULocationID", "hour_of_day")

with profiler.profile("O1-window-select"):
    flagged = (trips_typed
        .select("PULocationID", "payment_type", "fare_amount",
                "trip_distance", "tpep_pickup_datetime")
        .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
        .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
        .withColumn("is_weekend", F.dayofweek("tpep_pickup_datetime").isin(1, 7))
        .select("PULocationID", "payment_type", "hour_of_day", "is_weekend", "fare_per_mile")
        .withColumn("fpm_mean", F.avg("fare_per_mile").over(w))
        .withColumn("fpm_std",  F.stddev("fare_per_mile").over(w))
        .withColumn("is_anomaly", F.col("fare_per_mile") > F.col("fpm_mean") + 3 * F.col("fpm_std")))

    flagged = (flagged
        .join(F.broadcast(zones.select("LocationID", "Borough")),
              flagged.PULocationID == zones.LocationID, "left")
        .withColumnRenamed("Borough", "pickup_borough"))

    o1 = (flagged
        .groupBy("pickup_borough", "payment_type", "is_weekend")
        .agg(F.count("*").alias("trips"),
             F.avg(F.col("is_anomaly").cast("int")).alias("anomaly_rate")))

    o1.count()

o1.orderBy("pickup_borough", "payment_type", "is_weekend").show(50)
profiler.report()

[O1-window-select] wall time: 3.85s (7 stages, 22 tasks) - see profiler.report() / profiler.compare(...)
+--------------+------------+----------+-------+--------------------+
|pickup_borough|payment_type|is_weekend|  trips|        anomaly_rate|
+--------------+------------+----------+-------+--------------------+
|         Bronx|           1|     false|   5166|0.009697439875872769|
|         Bronx|           1|      true|   1747|0.009221902017291067|
|         Bronx|           2|     false|   5749|0.026067179846046187|
|         Bronx|           2|      true|   2333|0.023686477174849268|
|         Bronx|           3|     false|    102|  0.0784313725490196|
|         Bronx|           3|      true|     55| 0.07272727272727272|
|         Bronx|           4|     false|     45| 0.06666666666666667|
|         Bronx|           4|      true|     24|0.041666666666666664|
|      Brooklyn|           1|     false|  59410|0.006365243748421...|
|      Brooklyn|           1|      true|  25734| 0.0059

,wall_time_s,num_jobs,num_stages,num_tasks,executor_run_time_ms,executor_cpu_time_ms,input_bytes,output_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes
run,,,,,,,,,,,,
1-ingestion-cleaning,5.79,4,6,26,11937,9273,841.9KB,0.0B,696.0B,696.0B,0.0B,0.0B
2-temporal-demand-analysis,15.86,10,15,50,35844,33629,2.2MB,36.1KB,563.2KB,563.2KB,0.0B,0.0B
M3- Zone Level statistics,0.07,0,0,0,0,0,0.0B,0.0B,0.0B,0.0B,0.0B,0.0B
O1-fare-anomaly-window,4.41,4,7,22,10423,9857,753.0KB,0.0B,11.3KB,11.3KB,0.0B,0.0B
O1-fare-anomaly,8.35,7,14,59,30714,29377,1.5MB,0.0B,33.5MB,33.5MB,0.0B,0.0B
O1-fare-anomaly-persist,12.33,9,16,71,52875,44231,642.4MB,0.0B,33.5MB,33.5MB,0.0B,0.0B
O1-no-broadcast,9.74,7,24,112,39200,38030,1.5MB,0.0B,59.9MB,59.9MB,128.0MB,5.0MB
O1-partitions-6,9.28,2,6,26,33263,32488,1.5MB,0.0B,69.9MB,69.9MB,0.0B,0.0B
O1-partitions-12,7.95,2,6,38,29928,29222,1.5MB,0.0B,62.9MB,62.9MB,0.0B,0.0B


# O2 — Origin-destination flow analysis.
Compute monthly trip counts for all (PULocationID, DOLocationID) pairs. Join the zone
lookup to resolve zone names and boroughs.
Compute each route's share of total outbound trips from its pickup zone per month.
Identify the top-20 routes per pickup borough and detect routes whose monthly share reverses
direction between Q1 and Q3.

In [16]:
from pyspark.sql import Window

with profiler.profile("O2-od-flow"):
    od = (trips_typed
        .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime"))
        .groupBy("PULocationID", "DOLocationID", "month")
        .agg(F.count("*").alias("trips")))

    pu = zones.select(F.col("LocationID").alias("PULocationID"),
                      F.col("Borough").alias("pickup_borough"),
                      F.col("Zone").alias("pickup_zone"))
    do = zones.select(F.col("LocationID").alias("DOLocationID"),
                      F.col("Borough").alias("dropoff_borough"),
                      F.col("Zone").alias("dropoff_zone"))
    od = od.join(pu, "PULocationID", "left").join(do, "DOLocationID", "left")

    w_out = Window.partitionBy("PULocationID", "month")
    od = od.withColumn("share", F.col("trips") / F.sum("trips").over(w_out))

    w_rank = Window.partitionBy("pickup_borough", "month").orderBy(F.col("trips").desc())
    top20 = (od.withColumn("rank", F.row_number().over(w_rank))
               .filter(F.col("rank") <= 20))

    top20.count()

top20.orderBy("pickup_borough", "month", "rank").show(40, truncate=False)

[O2-od-flow] wall time: 4.17s (12 stages, 32 tasks) - see profiler.report() / profiler.compare(...)
+------------+------------+-------------------+-----+--------------+---------------------------------+---------------+---------------------------------+--------------------+----+
|DOLocationID|PULocationID|month              |trips|pickup_borough|pickup_zone                      |dropoff_borough|dropoff_zone                     |share               |rank|
+------------+------------+-------------------+-----+--------------+---------------------------------+---------------+---------------------------------+--------------------+----+
|78          |78          |2009-01-01 00:00:00|1    |Bronx         |East Tremont                     |Bronx          |East Tremont                     |1.0                 |1   |
|168         |168         |2020-01-01 00:00:00|194  |Bronx         |Mott Haven/Port Morris           |Bronx          |Mott Haven/Port Morris           |0.24556962025316456 |1   |
|74  